In [2]:
from prefect import flow, task
import requests
import pandas as pd
from pathlib import Path
import pandera.pandas as pa  # new import
from pandera import Column, DataFrameSchema, Check

from common.loaders import load_valid_country_ids

API_URL = "https://restcountries.com/v3.1/all?fields=cca3,idd"

# --- Esquema de validación ---
idd_schema = DataFrameSchema({
    "id_country": Column(
        str,
        [
            Check.str_length(3, 3),
            Check.isin(load_valid_country_ids())
        ],
        nullable=False
    ),
    "root": Column(str, nullable=True),
    "suffix": Column(str, nullable=True),
    "full_code": Column(str, nullable=True)
})

@task
def extract_idd():
    response = requests.get(API_URL)
    response.raise_for_status()
    return response.json()

@task
def transform_idd(data):
    df = pd.DataFrame(data)

    # Extraer root y suffixes
    df["root"] = df["idd"].apply(lambda x: x.get("root") if isinstance(x, dict) else None)
    df["suffixes"] = df["idd"].apply(lambda x: x.get("suffixes") if isinstance(x, dict) else [])

    # Expandir cada sufijo en filas separadas
    df_exploded = df.explode("suffixes").drop(columns=["idd"])

    # Crear el número telefónico completo
    df_exploded["full_code"] = df_exploded.apply(
        lambda row: f"{row['root']}{row['suffixes']}" if row["root"] and row["suffixes"] else None,
        axis=1
    )

    # Renombrar columnas
    df_exploded = df_exploded.rename(columns={"cca3": "id_country", "suffixes": "suffix"})
    return df_exploded

@task
def validate_idd(df: pd.DataFrame) -> pd.DataFrame:
    return idd_schema.validate(df)

@task
def load_idd(df: pd.DataFrame):
    file_path = Path("../stage/country_idd.csv")
    file_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(file_path, index=False, encoding="utf-8")
    print(f"Saved {len(df)} rows to {file_path}")

@flow(name="etl_idd_flow")
def etl_idd():
    data = extract_idd()
    df = transform_idd(data)
    df = validate_idd(df)
    load_idd(df)

if __name__ == "__main__":
    etl_idd()


C:\Users\Sebastian\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandera\_pandas_deprecated.py:149: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


04:09:15.445 | INFO    | Flow run 'heavy-cheetah' - Beginning flow run 'heavy-cheetah' for flow 'etl_idd_flow'

04:09:16.538 | INFO    | Task run 'extract_idd-8b6' - Finished in state Completed()

04:09:16.786 | INFO    | Task run 'transform_idd-4e2' - Finished in state Completed()

04:09:17.028 | INFO    | Task run 'validate_idd-914' - Finished in state Completed()

Saved 578 rows to ..\stage\country_idd.csv


04:09:17.251 | INFO    | Task run 'load_idd-046' - Finished in state Completed()

04:09:17.263 | INFO    | Flow run 'heavy-cheetah' - Finished in state Completed()